# **Data Cleaning Pipeline for Multiple Tabs or Sheets**

Luis Felipe Villota

InSight Crime - MAD Unit 

---------------------


In [114]:
import time

start_time = time.time()
formatted_start_time = time.ctime(start_time)
print(f"Start time: {formatted_start_time}")

Start time: Sat Apr  4 21:50:29 2026


## Setup

#### Version Control

The project is created within a single GitHub repository ([FelipeVillota/ic-prizes_and_awards](https://github.com/FelipeVillota/ic-prizes)). I keep the repository `private` with the possibility to give access to the online repo at any time. 

#### Environment

In [115]:
import os
import subprocess
import sys
import platform
from pathlib import Path

# ==============================
# CONFIG
# ==============================
VENV_NAME = "venv-cuina"
KERNEL_NAME = "cuina-kernel"
DISPLAY_NAME = "Python (cuina)"

PACKAGES = [
    "gspread", "pandas", "requests", "google-api-python-client",
    "oauth2client", "matplotlib", "seaborn", "missingno",
    "gspread-formatting", "ipykernel", "openpyxl",
    "gspread-dataframe", "pytz", "feedparser"
]

# ==============================
# PATH SETUP (CROSS PLATFORM)
# ==============================
project_dir = Path.cwd()
venv_path = project_dir / VENV_NAME

if platform.system() == "Windows":
    venv_python = venv_path / "Scripts" / "python.exe"
else:
    venv_python = venv_path / "bin" / "python"

# ==============================
# 1. CREATE VENV
# ==============================
if not venv_path.exists():
    print(f"🔧 Creating virtual environment: {VENV_NAME}")
    try:
        subprocess.run([sys.executable, "-m", "venv", str(venv_path)], check=True)
    except subprocess.CalledProcessError:
        print("❌ Failed to create virtual environment")
        sys.exit(1)
else:
    print(f"✅ Virtual environment already exists: {VENV_NAME}")

# Verify python exists
if not venv_python.exists():
    raise FileNotFoundError(f"❌ Python not found in venv: {venv_python}")

# ==============================
# 2. ENSURE PIP
# ==============================
subprocess.run([str(venv_python), "-m", "ensurepip", "--upgrade"], check=False)

# Upgrade pip safely
subprocess.run([str(venv_python), "-m", "pip", "install", "--upgrade", "pip"], check=True)

# ==============================
# 3. CHECK INSTALLED PACKAGES
# ==============================
try:
    result = subprocess.run(
        [str(venv_python), "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True
    )
    installed = {
        line.split("==")[0].lower()
        for line in result.stdout.splitlines()
        if "==" in line
    }
except subprocess.CalledProcessError:
    installed = set()
    print("⚠️ Could not read installed packages")

missing = [pkg for pkg in PACKAGES if pkg.lower() not in installed]

# ==============================
# 4. INSTALL PACKAGES
# ==============================
if missing:
    print(f"📦 Installing missing packages: {missing}")
    subprocess.run(
        [str(venv_python), "-m", "pip", "install"] + missing,
        check=True
    )
    print("✅ Packages installed")
else:
    print("✅ All packages already installed")

# ==============================
# 5. REGISTER JUPYTER KERNEL
# ==============================
try:
    subprocess.run([
        str(venv_python), "-m", "ipykernel", "install",
        "--user",
        "--name", KERNEL_NAME,
        "--display-name", DISPLAY_NAME
    ], check=True)
    print(f"✅ Jupyter kernel registered: {DISPLAY_NAME}")
except subprocess.CalledProcessError:
    print("❌ Failed to register Jupyter kernel")
    sys.exit(1)

# ==============================
# 6. EXPORT REQUIREMENTS
# ==============================
requirements_path = project_dir / "requirements.txt"

with open(requirements_path, "w") as f:
    subprocess.run(
        [str(venv_python), "-m", "pip", "freeze"],
        stdout=f,
        check=True
    )

print(f"✅ requirements.txt created at {requirements_path}")

# ==============================
# DONE
# ==============================
print(f"🐍 Python executable: {venv_python}")

✅ Virtual environment already exists: venv-cuina
✅ All packages already installed
✅ Jupyter kernel registered: Python (cuina)
✅ requirements.txt created at c:\Users\lfvm0\Desktop\ic-audiences\requirements.txt
🐍 Python executable: c:\Users\lfvm0\Desktop\ic-audiences\venv-cuina\Scripts\python.exe


In [116]:
import os
from datetime import datetime

# Get current date
print("Current Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Count items, files, and directories
items = os.listdir()
files = [f for f in items if os.path.isfile(f)]
directories = [d for d in items if os.path.isdir(d)]

print(f"Total Items: {len(items)}")
print(f"Files: {len(files)}")
print(f"Directories: {len(directories)}")

# Get files sorted by modification time 
print("\nJust files (newest first):")
files_with_mtime = [(f, os.path.getmtime(f)) for f in items if os.path.isfile(f)]
sorted_files = sorted(files_with_mtime, key=lambda x: x[1], reverse=True)

for file, mtime in sorted_files:
    print(f"{datetime.fromtimestamp(mtime)} - {file}")

Current Date: 2026-04-04 21:50:40
Total Items: 6
Files: 3
Directories: 3

Just files (newest first):
2026-04-04 21:50:40.107377 - requirements.txt
2026-04-04 21:50:38.461626 - initial_config.ipynb
2026-04-03 14:54:37.332517 - .gitignore


#### Libraries

In [117]:
import re
import requests
import pandas as pd
from datetime import datetime
from google.oauth2 import service_account
from googleapiclient.discovery import build
import gspread
from google.oauth2.service_account import Credentials
from gspread_formatting import format_cell_ranges, CellFormat, Color
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import openpyxl as px
import gspread_dataframe as gd

### API

This creates a modular client (frontend) call that is able to extract the desired subset of data from the API server (backend); -and, make it easily reusable for future queries.


In [118]:
# Path to your service account key file
SERVICE_ACCOUNT_FILE = r'C:\Users\lfvm0\Desktop\summer-sector-439022-v6-0988365c484e.json'

# '/home/lfvm/codenivore/codebaker/nyckelring/brelok/summer-sector-439022-v6-2eafffbbfb90.json' # Update with the actual path or team credentials file

# VERY IMPORTANT: Ensure the service account has access to the Google Sheet by sharing it with the service account email

# Original Google Sheet ID (latest version) 
ORIGINAL_SPREADSHEET_ID =  '1x9q9MJcHynAjr14CdvygFFDUWoHRM1Y0qtoIMrgkwHg'

# Define scopes for Google Sheets and Drive API
SCOPES = ['https://www.googleapis.com/auth/spreadsheets', 
          'https://www.googleapis.com/auth/drive']

GOOGLE_API_KEY = '2eafffbbfb905d4cef6a81b4bff19cfabf7adf0b'
GOOGLE_CX = '65163f283105448fa'

In [119]:
# Authenticate and build both Sheets and Drive services
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
sheet_service = build('sheets', 'v4', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

#### First call

In [120]:
# Original spreadsheet
# Last modification: user and time. This information is related to the file itself, not to specific content (sheets, cells, rows)

from googleapiclient.discovery import build
from datetime import datetime
import pytz

def get_last_modifying_user(drive_service, file_id):
    try:
        # Get revisions (only the last one)
        revisions = drive_service.revisions().list(
            fileId=file_id,
            fields="revisions(modifiedTime,lastModifyingUser)"
        ).execute().get('revisions', [])
        
        if not revisions:
            return "No revisions found."
        
        last_revision = revisions[-1]  # Most recent revision
        user_info = last_revision.get('lastModifyingUser', {})
        email = user_info.get('emailAddress', 'Unknown')
        modified_time = last_revision.get('modifiedTime', 'Unknown')
        
        # Convert to readable datetime
        if modified_time != 'Unknown':
            dt = datetime.strptime(modified_time, "%Y-%m-%dT%H:%M:%S.%fZ")
            dt = dt.replace(tzinfo=pytz.UTC)
            modified_time = dt.strftime("%Y-%m-%d %H:%M:%S (UTC)")
        
        return {
            "user_email": email,
            "modified_time": modified_time
        }
    
    except Exception as e:
        return f"Error: {str(e)}"

# Usage
# First get the file name and sheet name
file_metadata = drive_service.files().get(fileId=ORIGINAL_SPREADSHEET_ID, fields='name').execute()
file_name = file_metadata.get('name', 'Unknown')

# Get sheet names
sheets = sheet_service.spreadsheets().get(
    spreadsheetId=ORIGINAL_SPREADSHEET_ID,
    fields='sheets(properties(title))'
).execute().get('sheets', [])

sheet_names = [sheet['properties']['title'] for sheet in sheets]

# Get last modifier info
last_modifier = get_last_modifying_user(drive_service, ORIGINAL_SPREADSHEET_ID)


print(
    f"📄 File name: {file_name}\n"
    f"📑 Sheet names: {', '.join(sheet_names)}\n"
    f"🔄 Last modified by: {last_modifier['user_email']}\n"
    f"⏰ Last modified at: {last_modifier['modified_time']}"
)

📄 File name: mailchimp-clean-upload
📑 Sheet names: ONGS, Argentina , Homicidios, Drogas Sinteticas, Pandillas, Tennesse media, Migration (US civil society and instutional), Ambiente, Narcotrafico, Esequibo,  Venezuela, Elites y Crimen, Género, Paz Colombia, US terrorism event - Civil, US terrorism event - Govt
🔄 Last modified by: lvillota@insightcrime.org
⏰ Last modified at: 2026-04-05 02:09:18 (UTC)


## Extraction


### Tab names

In [121]:
spreadsheet_metadata = sheet_service.spreadsheets().get(
    spreadsheetId=ORIGINAL_SPREADSHEET_ID
).execute()

sheets = spreadsheet_metadata.get('sheets', [])
sheet_names = [s['properties']['title'] for s in sheets]

print(sheet_names)

['ONGS', 'Argentina ', 'Homicidios', 'Drogas Sinteticas', 'Pandillas', 'Tennesse media', 'Migration (US civil society and instutional)', 'Ambiente', 'Narcotrafico', 'Esequibo', ' Venezuela', 'Elites y Crimen', 'Género', 'Paz Colombia', 'US terrorism event - Civil', 'US terrorism event - Govt']


### Import and summarize

In [122]:
import pandas as pd

sheets_dict = {}

for sheet_name in sheet_names:
    result = sheet_service.spreadsheets().values().get(
        spreadsheetId=ORIGINAL_SPREADSHEET_ID,
        range=sheet_name
    ).execute()
    
    values = result.get('values', [])
    
    if not values or len(values) < 2:
        print(f"[SKIP] {sheet_name}: insufficient data (rows: {len(values)})")
        continue
    
    headers = values[0]
    rows = values[1:]
    
    # --- FIX: normalize row lengths ---
    max_len = max(len(headers), max(len(r) for r in rows))
    
    # expand headers if needed
    headers = headers + [f"extra_col_{i}" for i in range(len(headers), max_len)]
    
    fixed_rows = []
    for i, row in enumerate(rows):
        if len(row) != len(headers):
            print(f"[WARNING] {sheet_name} row {i} has {len(row)} cols, expected {len(headers)}")
        
        row = row + [""] * (len(headers) - len(row))  # pad missing
        fixed_rows.append(row[:len(headers)])         # trim overflow
    
    df = pd.DataFrame(fixed_rows, columns=headers)
    
    # --- clean columns ---
    df.columns = df.columns.str.strip().str.lower()
    
    # --- add metadata ---
    df["source_sheet"] = sheet_name
    df["theme"] = sheet_name
    
    sheets_dict[sheet_name] = df

# --- PRODUCE DIMENSIONS AND COLUMNS TABLE ---
print("\n" + "="*80)
print("TABLE: Sheet Dimensions and Column Names")
print("="*80)

# Create a list to store the table data
table_data = []

for sheet_name, df in sheets_dict.items():
    rows, cols = df.shape
    column_list = ", ".join(df.columns.tolist())
    
    table_data.append({
        "Sheet Name": sheet_name,
        "Rows": rows,
        "Columns": cols,
        "Column Names": column_list
    })

# Create DataFrame for display
summary_df = pd.DataFrame(table_data)

# Display the table
print(summary_df.to_string(index=False))

# Optional: Save to CSV
# summary_df.to_csv("sheet_dimensions_and_columns.csv", index=False)
# print("\n[SAVED] Table exported to 'sheet_dimensions_and_columns.csv'")

# Optional: Display detailed view per sheet
print("\n" + "="*80)
print("DETAILED VIEW PER SHEET")
print("="*80)

for sheet_name, df in sheets_dict.items():
    print(f"\n📊 {sheet_name}")
    print(f"   Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"   Columns ({len(df.columns)}):")
    for i, col in enumerate(df.columns, 1):
        print(f"      {i}. {col}")
    print("-"*50)

[WARNING] ONGS row 0 has 8 cols, expected 10
[WARNING] ONGS row 1 has 8 cols, expected 10
[WARNING] ONGS row 2 has 8 cols, expected 10
[WARNING] ONGS row 3 has 8 cols, expected 10
[WARNING] ONGS row 4 has 8 cols, expected 10
[WARNING] ONGS row 5 has 8 cols, expected 10
[WARNING] ONGS row 6 has 8 cols, expected 10
[WARNING] ONGS row 7 has 8 cols, expected 10
[WARNING] ONGS row 8 has 8 cols, expected 10
[WARNING] ONGS row 9 has 8 cols, expected 10
[WARNING] ONGS row 10 has 8 cols, expected 10
[WARNING] ONGS row 11 has 8 cols, expected 10
[WARNING] ONGS row 12 has 8 cols, expected 10
[WARNING] ONGS row 13 has 8 cols, expected 10
[WARNING] ONGS row 14 has 8 cols, expected 10
[WARNING] ONGS row 15 has 8 cols, expected 10
[WARNING] ONGS row 16 has 8 cols, expected 10
[WARNING] ONGS row 17 has 8 cols, expected 10
[WARNING] ONGS row 18 has 8 cols, expected 10
[WARNING] ONGS row 19 has 8 cols, expected 10
[WARNING] ONGS row 20 has 8 cols, expected 10
[WARNING] ONGS row 21 has 8 cols, expected 1

## Transformation

### Boundaries

In [123]:
def trim_sheet_to_data(sheet_service, spreadsheet_id, sheet_name):
    """Remove blank rows and columns after the end of actual data in the sheet"""
    
    # Get sheet details without grid data
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(sheetId,title,gridProperties(rowCount,columnCount)))"
    ).execute()

    sheet_id = None
    total_row_count = 0
    total_col_count = 0
    
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] == sheet_name:
            sheet_id = sheet["properties"]["sheetId"]
            total_row_count = sheet["properties"]["gridProperties"]["rowCount"]
            total_col_count = sheet["properties"]["gridProperties"]["columnCount"]
            break

    if sheet_id is None:
        raise ValueError(f"Sheet '{sheet_name}' not found")

    print(f"\n{'='*60}")
    print(f"TRIMMING TAB: '{sheet_name}'")
    print(f"{'='*60}")
    print(f"Current dimensions: {total_row_count} rows × {total_col_count} columns")
    
    # Get all data at once for more efficient column scanning
    try:
        result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_name,
            majorDimension="ROWS"
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print(f"No data found in sheet '{sheet_name}'. No trimming needed.")
            return total_row_count, total_col_count
        
        # Find last row with data
        last_data_row = 0
        for row_idx in range(len(values) - 1, -1, -1):
            row = values[row_idx]
            has_data = any(cell and str(cell).strip() for cell in row)
            if has_data:
                last_data_row = row_idx + 1  # Convert to 1-based index
                break
        
        # Find last column with data
        last_data_col = 0
        # Transpose to scan by columns
        max_cols = max(len(row) for row in values) if values else 0
        for col_idx in range(max_cols - 1, -1, -1):
            has_data = False
            for row in values:
                if col_idx < len(row) and row[col_idx] and str(row[col_idx]).strip():
                    has_data = True
                    break
            if has_data:
                last_data_col = col_idx + 1  # Convert to 1-based index
                break
        
        # Prepare batch update requests
        requests = []
        
        # Delete blank rows if needed
        if last_data_row < total_row_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": last_data_row,
                        "endIndex": total_row_count
                    }
                }
            })
        
        # Delete blank columns if needed
        if last_data_col < total_col_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": last_data_col,
                        "endIndex": total_col_count
                    }
                }
            })
        
        # Execute batch update if there are changes
        if requests:
            delete_request = {"requests": requests}
            
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body=delete_request
            ).execute()
            
            rows_trimmed = total_row_count - last_data_row if last_data_row < total_row_count else 0
            cols_trimmed = total_col_count - last_data_col if last_data_col < total_col_count else 0
            
            print(f"✓ Trimmed {rows_trimmed} blank rows and {cols_trimmed} blank columns from tab '{sheet_name}'.")
            print(f"✓ New dimensions: {last_data_row} rows × {last_data_col} columns")
            
            return last_data_row, last_data_col
        else:
            print(f"✓ No blank rows or columns to trim in tab '{sheet_name}'.")
            return total_row_count, total_col_count
            
    except Exception as e:
        print(f"✗ Error processing tab '{sheet_name}': {e}")
        return total_row_count, total_col_count


def trim_all_tabs(sheet_service, spreadsheet_id):
    """Trim blank rows and columns from ALL tabs in the spreadsheet"""
    
    # Get all sheet names
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(title))"
    ).execute()
    
    sheet_names = [sheet["properties"]["title"] for sheet in spreadsheet["sheets"]]
    
    print(f"\n{'#'*60}")
    print(f"TRIMMING ALL TABS IN SPREADSHEET")
    print(f"{'#'*60}")
    print(f"Found {len(sheet_names)} tab(s): {', '.join(sheet_names)}")
    
    results = []
    for sheet_name in sheet_names:
        rows, cols = trim_sheet_to_data(sheet_service, spreadsheet_id, sheet_name)
        results.append({
            'Tab Name': sheet_name,
            'Final Rows': rows,
            'Final Columns': cols
        })
    
    # Print summary of all tabs
    print(f"\n{'='*60}")
    print("SUMMARY - ALL TABS")
    print(f"{'='*60}")
    for result in results:
        print(f"  Tab '{result['Tab Name']}': {result['Final Rows']} rows × {result['Final Columns']} columns")
    print(f"{'='*60}")
    
    return results


# Usage - Trim ALL tabs:
trim_all_tabs(sheet_service, ORIGINAL_SPREADSHEET_ID)

# OR trim a single specific tab:
# trim_sheet_to_data(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")


############################################################
TRIMMING ALL TABS IN SPREADSHEET
############################################################
Found 16 tab(s): ONGS, Argentina , Homicidios, Drogas Sinteticas, Pandillas, Tennesse media, Migration (US civil society and instutional), Ambiente, Narcotrafico, Esequibo,  Venezuela, Elites y Crimen, Género, Paz Colombia, US terrorism event - Civil, US terrorism event - Govt

TRIMMING TAB: 'ONGS'
Current dimensions: 213 rows × 10 columns
✓ No blank rows or columns to trim in tab 'ONGS'.

TRIMMING TAB: 'Argentina '
Current dimensions: 29 rows × 9 columns
✓ No blank rows or columns to trim in tab 'Argentina '.

TRIMMING TAB: 'Homicidios'
Current dimensions: 239 rows × 9 columns
✓ No blank rows or columns to trim in tab 'Homicidios'.

TRIMMING TAB: 'Drogas Sinteticas'
Current dimensions: 323 rows × 12 columns
✓ No blank rows or columns to trim in tab 'Drogas Sinteticas'.

TRIMMING TAB: 'Pandillas'
Current dimensions: 305 rows × 14 co

[{'Tab Name': 'ONGS', 'Final Rows': 213, 'Final Columns': 10},
 {'Tab Name': 'Argentina ', 'Final Rows': 29, 'Final Columns': 9},
 {'Tab Name': 'Homicidios', 'Final Rows': 239, 'Final Columns': 9},
 {'Tab Name': 'Drogas Sinteticas', 'Final Rows': 323, 'Final Columns': 12},
 {'Tab Name': 'Pandillas', 'Final Rows': 305, 'Final Columns': 14},
 {'Tab Name': 'Tennesse media', 'Final Rows': 33, 'Final Columns': 4},
 {'Tab Name': 'Migration (US civil society and instutional)',
  'Final Rows': 34,
  'Final Columns': 3},
 {'Tab Name': 'Ambiente', 'Final Rows': 493, 'Final Columns': 16},
 {'Tab Name': 'Narcotrafico', 'Final Rows': 128, 'Final Columns': 9},
 {'Tab Name': 'Esequibo', 'Final Rows': 442, 'Final Columns': 13},
 {'Tab Name': ' Venezuela', 'Final Rows': 278, 'Final Columns': 13},
 {'Tab Name': 'Elites y Crimen', 'Final Rows': 161, 'Final Columns': 10},
 {'Tab Name': 'Género', 'Final Rows': 433, 'Final Columns': 21},
 {'Tab Name': 'Paz Colombia', 'Final Rows': 401, 'Final Columns': 14},

### Email detection and cleaning

* Scans all columns in each row for email patterns
* Fills email if reference column is blank and there is an email in that row anywhere else (first is selected)
* Replaces email if it contains non-email text in the reference column and there is an email in that row anywhere else (first is selected)
* Keeps valid emails unchanged
* Normalizes emails
* Removes duplicates
* Removes rows with missing emails
* Removes rows that are all blank
* Logs duplicates and summaries

In [ ]:
import re
import pandas as pd

# --- Email helpers ---
EMAIL_REGEX = r'[\w\.-]+@[\w\.-]+\.\w+'

def extract_email_from_row(row):
    for val in row:
        if isinstance(val, str):
            match = re.search(EMAIL_REGEX, val)
            if match:
                return match.group(0)
    return None

def is_valid_email(val):
    if not isinstance(val, str):
        return False
    return bool(re.match(f"^{EMAIL_REGEX}$", val.strip()))

# --- Storage ---
cleaned_dfs = []
summary_stats = []
duplicate_log = []

# --- Process each sheet ---
for sheet_name, file in sheets_dict.items():
    # Read with proper header handling
    df = pd.read_excel(file, header=0)
    df = df.copy()
    
    original_rows = len(df)
    
    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()
    
    # --- Remove completely blank rows ---
    blank_mask = df.isna().all(axis=1)
    blank_removed = df[blank_mask]
    df = df[~blank_mask]
    
    # Ensure email column exists
    if "email" not in df.columns:
        df["email"] = None
    
    # --- Extract / fix emails ---
    df["email"] = df.apply(
        lambda row: row["email"] if is_valid_email(row["email"])
        else extract_email_from_row(row),
        axis=1
    )
    
    # Normalize emails
    df["email"] = df["email"].where(df["email"].notna(), None)
    df["email"] = df["email"].astype(str).str.lower().str.strip()
    df.loc[df["email"] == "none", "email"] = None
    
    # --- Count & remove rows with NO email ---
    no_email_mask = df["email"].isna()
    no_email_removed = df[no_email_mask]
    df = df[~no_email_mask]
    
    # --- Detect duplicates ---
    dup_mask = df.duplicated(subset=["email"], keep="first")
    duplicates = df[dup_mask]
    
    # Log duplicates
    for idx, row in duplicates.iterrows():
        duplicate_log.append({
            "Tab": sheet_name,
            "Row Index Removed": idx,
            "Duplicate Email": row["email"]
        })
    
    # Remove duplicates
    df = df[~dup_mask]
    
    cleaned_dfs.append(df)
    
    # --- Summary stats ---
    summary_stats.append({
        'Tab Name': sheet_name,
        'Original Rows': original_rows,
        'Blank Rows Removed': len(blank_removed),
        'No Email Rows Removed': len(no_email_removed),
        'Duplicate Rows Removed': len(duplicates),
        'Final Rows': len(df),
        'Unique Emails': df['email'].nunique()
    })
    
    # --- Print summary ---
    print(f"\n{'='*50}")
    print(f"SUMMARY FOR TAB: '{sheet_name}'")
    print(f"{'='*50}")
    print(f"  Original rows:           {original_rows}")
    print(f"  Blank rows removed:      {len(blank_removed)}")
    print(f"  No-email rows removed:   {len(no_email_removed)}")
    print(f"  Duplicate rows removed:  {len(duplicates)}")
    print(f"  Final rows:              {len(df)}")
    print(f"  Unique emails:           {df['email'].nunique()}")
    print(f"{'='*50}")

# --- Duplicate log output ---
print(f"\n{'='*60}")
print("DUPLICATES REMOVED (DETAIL)")
print(f"{'='*60}")
dup_df = pd.DataFrame(duplicate_log)
print(dup_df.to_string(index=False))

# --- Overall summary ---
print(f"\n{'='*60}")
print("OVERALL SUMMARY (All Tabs)")
print(f"{'='*60}")
summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))
print(f"{'='*60}")


SUMMARY FOR TAB: 'ONGS'
  Original rows:           212
  Blank rows removed:      0
  No-email rows removed:   22
  Duplicate rows removed:  36
  Final rows:              154
  Unique emails:           154

SUMMARY FOR TAB: 'Argentina '
  Original rows:           28
  Blank rows removed:      0
  No-email rows removed:   3
  Duplicate rows removed:  0
  Final rows:              25
  Unique emails:           25

SUMMARY FOR TAB: 'Homicidios'
  Original rows:           238
  Blank rows removed:      0
  No-email rows removed:   28
  Duplicate rows removed:  53
  Final rows:              157
  Unique emails:           157

SUMMARY FOR TAB: 'Drogas Sinteticas'
  Original rows:           322
  Blank rows removed:      0
  No-email rows removed:   1
  Duplicate rows removed:  6
  Final rows:              315
  Unique emails:           315

SUMMARY FOR TAB: 'Pandillas'
  Original rows:           304
  Blank rows removed:      0
  No-email rows removed:   1
  Duplicate rows removed:  64
  Fin

#### Looking at each cleaned DataFrame

In [ ]:

for i, df in enumerate(cleaned_dfs):
    print(f"\n{'='*60}")
    print(f"DataFrame {i} (Shape: {df.shape})")
    print(f"{'='*60}")
    print(df.head(10))  # First 10 rows
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nEmail sample: {df['email'].head().tolist()}")

In [ ]:
# See all emails from all cleaned DataFrames
for i, df in enumerate(cleaned_dfs):
    print(f"\n📧 Sheet {i} - Found {len(df)} emails:")
    print(df['email'].tolist())  # All emails as a list

In [ ]:
# Show emails with their original rows
for i, df in enumerate(cleaned_dfs):
    print(f"\n{'='*60}")
    print(f"Sheet {i} - {len(df)} emails detected")
    print(f"{'='*60}")
    
    # Show email and first few other columns
    cols_to_show = ['email'] + [c for c in df.columns if c not in ['email', 'source_sheet', 'theme']][:3]
    print(df[cols_to_show].head(20))  # First 20 rows with emails

### Sectors: Explore & Handle

### Unique sectors by Tab

In [ ]:
sector_summary = []

for sheet_name, df in sheets_dict.items():
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    
    if "sector" in df.columns:
        unique_sectors = (
            df["sector"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )
        
        if len(unique_sectors) == 0:
            unique_sectors = ["empty"]
    else:
        unique_sectors = ["Does not have"]
    
    sector_summary.append({
        "sheet": sheet_name,
        "unique_sectors": unique_sectors,
        "n_unique": len(unique_sectors)
    })

sector_table = pd.DataFrame(sector_summary)

sector_table

In [ ]:
# names of all unique values in the unique_sectors column
all_unique_sectors = set()  
for _, row in sector_table.iterrows():
    all_unique_sectors.update(row["unique_sectors"]) 

print(all_unique_sectors)   

### Map and clean sectors

In [ ]:
sector_mapping = {
    # --- Government ---
    "gobierno": "Government",
    "Gobierno": "Government",
    "Government": "Government",
    "Ex gobierno": "Government",
    "Embajadas": "Government",

    # --- Private Sector ---
    "Sector Privado": "Private Sector",
    "Sector privado": "Private Sector",
    "Privado": "Private Sector",
    "Consultoría": "Private Sector",
    "Análisis de riesgo y/o seguridad privada": "Private Sector",

   # --- Academia / Think Tank ---
    "Academia": "Academia / Think Tank",
    "Think Tank": "Academia / Think Tank",
    "Think tank": "Academia / Think Tank",
    "Centro de pensamiento": "Academia / Think Tank",
    "Estudiante": "Academia / Think Tank",

    # --- Civil Society ---
    "Sociedad civil": "Civil Society",
    "Sociedad Civil": "Civil Society",
    "Civil Society": "Civil Society",
    "ONG": "Civil Society",
    "NGO": "Civil Society",
    "No Gubernamental (ONG)": "Civil Society",

    # --- Media ---
    "medio de comunicación": "Media",
    "Medio de comunicación": "Media",
    "Medio de Comunicación": "Media",
    "Medio de comunicaicón": "Media",  # typo fixed
    "Medios": "Media",
    "Periodistas": "Media",

    # --- International Organizations ---
    "Organismo internacional": "International Organization",
    "Organismo Internacional": "International Organization",
    "Organización internacional": "International Organization",
    "Organismo Organismo Internacional": "International Organization",

    # --- Unknown / garbage ---
    "No aplica": "Unknown",
    "No especificado": "Unknown",
    "Desconocido": "Unknown",
    "N/A": "Unknown",
    "dont have": "Unknown",
    "Sector": "Unknown",

    # --- Catch-all ---
    "Otro": "Other"
}

def normalize_sector(value):
    if pd.isna(value):
        return "Unknown"
    
    value = str(value).strip()
    
    return sector_mapping.get(value, "Other")

df["sector_clean"] = df["sector"].apply(normalize_sector)

In [ ]:
print(df["sector_clean"].value_counts())

In [ ]:
unmapped = df[
    ~df["sector"].isin(sector_mapping.keys())
]["sector"].unique()

print(unmapped)

### Combine sheets

In [ ]:
cleaned_dfs_fixed = []

for i, df in enumerate(cleaned_dfs):
    df = df.copy()
    
    # Ensure columns are strings + normalized
    df.columns = df.columns.astype(str).str.strip().str.lower()
    
    # Ensure uniqueness again (critical)
    df.columns = make_unique_columns(df.columns)
    
    # Reset index (extra safety)
    df = df.reset_index(drop=True)
    
    cleaned_dfs_fixed.append(df)

# CONCAT safely
master_df = pd.concat(cleaned_dfs_fixed, ignore_index=True)

#### Dropdowns


###### Options

In [ ]:
# Indentify Existing Dropdowns
 
def get_sheet_dropdowns(spreadsheet_id, sheet_name, creds):
    service = build('sheets', 'v4', credentials=creds)
    sheet_metadata = service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        ranges=[sheet_name],
        includeGridData=True
    ).execute()

    dropdowns_by_column = {}

    for sheet in sheet_metadata['sheets']:
        if sheet['properties']['title'] != sheet_name:
            continue

        col_dropdowns = {}
        rows = sheet['data'][0]['rowData']
        header_row = rows[0]['values']
        col_names = [cell.get('formattedValue', f"Column_{i}") for i, cell in enumerate(header_row)]

        for col_idx, col_name in enumerate(col_names):
            dropdowns_in_col = set()
            for row in rows[1:]:  # Skip header
                cell = row['values'][col_idx]
                if 'dataValidation' in cell:
                    dv = cell['dataValidation']
                    if dv['condition']['type'] == 'ONE_OF_LIST':
                        values = [v['userEnteredValue'] for v in dv['condition']['values']]
                        dropdowns_in_col.update(values)
            if dropdowns_in_col:
                col_dropdowns[col_name] = list(dropdowns_in_col)

        dropdowns_by_column.update(col_dropdowns)

    return dropdowns_by_column

dropdown_options = get_sheet_dropdowns(ORIGINAL_SPREADSHEET_ID, "aligned", creds)

# Print results
for col, options in dropdown_options.items():
    print(f"🔽 {col}: {options}")


In [ ]:
dropdown_options = {

    
    "Status": [
        "Open", 
        "Closed"
        
    ],

     "Decision": [
        "New Entry",
        "Apply",
        "Maybe",
        "Skip",
        "Unknown"
    ],
    
    
"Category": [
    "Investigative Journalism",
    "Cross-Border Investigation",
    "Documentary / Film",
    "Podcast / Audio",
    "Data / Visualization",
    "Research / Academic",
    "NGO / Advocacy",
    "Security / Defense",
    "Other"
]

}

In [ ]:
def apply_dropdowns_to_sheet(sheet_service, spreadsheet_id, sheet_name, dropdown_options, max_rows=150000):
    
    # Get the sheet ID by matching the sheet name
    spreadsheet = sheet_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
    sheet_id = None
    for sheet in spreadsheet['sheets']:
        if sheet['properties']['title'] == sheet_name:
            sheet_id = sheet['properties']['sheetId']
            break
    if sheet_id is None:
        print(f"❌ Sheet name '{sheet_name}' not found.")
        return

    # Fetch header row (assumes headers are in row 1)
    result = sheet_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"{sheet_name}!1:1"
    ).execute()
    headers = result.get('values', [])[0]

    requests = []  # Will store all the API requests we need to make

    # Handle both direct list options and dictionary format with "official_list"
    for col_name, options in dropdown_options.items():
        if col_name not in headers:
            continue

        col_index = headers.index(col_name)
        values = (
            options.get("official_list", []) if isinstance(options, dict)
            else options
        )
        values = values[:500]  # Apparently it can only handle this, or else it breaks the API

        if not values:
            continue

    # Create data validation rule for dropdown
        validation_rule = {
            "condition": {
                "type": "ONE_OF_LIST",
                "values": [{"userEnteredValue": str(v)} for v in values]
            },
            "strict": True,
            "showCustomUi": True
        }

    # Add request for this column's dropdown
        requests.append({
            "setDataValidation": {
                "range": {
                    "sheetId": sheet_id,
                    "startRowIndex": 1,
                    "endRowIndex": max_rows,
                    "startColumnIndex": col_index,
                    "endColumnIndex": col_index + 1
                },
                "rule": validation_rule
            }
        })

    # Execute all requests if we have any
    if requests:
        try:
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body={"requests": requests}
            ).execute()
            print(f"✅ Applied dropdown formatting to {len(requests)} columns.")
        except Exception as e:
            print(f"❌ Error applying dropdowns: {e}")
    else:
        print("⚠️ No matching columns found in the sheet.")
        
apply_dropdowns_to_sheet(
    sheet_service=sheet_service,
    spreadsheet_id=ORIGINAL_SPREADSHEET_ID,  
    sheet_name="aligned",                 
    dropdown_options=dropdown_options, 
    max_rows=150000                  
)


In [ ]:
dropdown_options['Status']

#### Boundaries

Trim blank rows and columns to have the exact db size.

In [ ]:
def trim_sheet_to_data(sheet_service, spreadsheet_id, sheet_name):
    """Remove blank rows and columns after the end of actual data in the sheet"""
    
    # Get sheet details without grid data
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(sheetId,title,gridProperties(rowCount,columnCount)))"
    ).execute()

    sheet_id = None
    total_row_count = 0
    total_col_count = 0
    
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] == sheet_name:
            sheet_id = sheet["properties"]["sheetId"]
            total_row_count = sheet["properties"]["gridProperties"]["rowCount"]
            total_col_count = sheet["properties"]["gridProperties"]["columnCount"]
            break

    if sheet_id is None:
        raise ValueError(f"Sheet '{sheet_name}' not found")

    print(f"Scanning sheet '{sheet_name}' for data boundaries...")
    
    # Get all data at once for more efficient column scanning
    try:
        result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_name,
            majorDimension="ROWS"
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print(f"No data found in sheet '{sheet_name}'. No trimming needed.")
            return total_row_count, total_col_count
        
        # Find last row with data
        last_data_row = 0
        for row_idx in range(len(values) - 1, -1, -1):
            row = values[row_idx]
            has_data = any(cell and str(cell).strip() for cell in row)
            if has_data:
                last_data_row = row_idx + 1  # Convert to 1-based index
                break
        
        # Find last column with data
        last_data_col = 0
        # Transpose to scan by columns
        max_cols = max(len(row) for row in values) if values else 0
        for col_idx in range(max_cols - 1, -1, -1):
            has_data = False
            for row in values:
                if col_idx < len(row) and row[col_idx] and str(row[col_idx]).strip():
                    has_data = True
                    break
            if has_data:
                last_data_col = col_idx + 1  # Convert to 1-based index
                break
        
        # Prepare batch update requests
        requests = []
        
        # Delete blank rows if needed
        if last_data_row < total_row_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": last_data_row,
                        "endIndex": total_row_count
                    }
                }
            })
        
        # Delete blank columns if needed
        if last_data_col < total_col_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": last_data_col,
                        "endIndex": total_col_count
                    }
                }
            })
        
        # Execute batch update if there are changes
        if requests:
            delete_request = {"requests": requests}
            
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body=delete_request
            ).execute()
            
            rows_trimmed = total_row_count - last_data_row if last_data_row < total_row_count else 0
            cols_trimmed = total_col_count - last_data_col if last_data_col < total_col_count else 0
            
            print(f"Trimmed {rows_trimmed} blank rows and {cols_trimmed} blank columns from sheet '{sheet_name}'.")
            print(f"New dimensions: {last_data_row} rows × {last_data_col} columns")
            
            return last_data_row, last_data_col
        else:
            print(f"No blank rows or columns to trim in sheet '{sheet_name}'.")
            return total_row_count, total_col_count
            
    except Exception as e:
        print(f"Error processing sheet '{sheet_name}': {e}")
        return total_row_count, total_col_count


# Usage
trim_sheet_to_data(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")

#### Drop blanks

In [ ]:
def drop_blank_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop rows that are entirely blank (all values are NaN or empty string).
    
    Args:
        df: Input DataFrame
    
    Returns:
        DataFrame without blank rows
    """
    df_cleaned = df.replace("", pd.NA).dropna(how="all").reset_index(drop=True)
    return df_cleaned

df_target = drop_blank_rows(df_target)


In [ ]:
def delete_all_blank_rows(sheet_service, spreadsheet_id, sheet_name):
    """Delete all completely blank rows within the data area of the sheet"""
    
    # Get sheet details
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(sheetId,title))"
    ).execute()

    sheet_id = None
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] == sheet_name:
            sheet_id = sheet["properties"]["sheetId"]
            break

    if sheet_id is None:
        raise ValueError(f"Sheet '{sheet_name}' not found")

    print(f"Scanning sheet '{sheet_name}' for blank rows...")
    
    # Get all data
    try:
        result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_name,
            majorDimension="ROWS"
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print(f"No data found in sheet '{sheet_name}'. No rows to delete.")
            return 0
        
        # Identify blank rows (bottom-up to maintain index integrity)
        blank_row_indices = []
        for row_idx in range(len(values) - 1, -1, -1):
            row = values[row_idx]
            # Check if row is completely empty or has only empty strings
            is_blank = True
            for cell in row:
                if cell and str(cell).strip():
                    is_blank = False
                    break
            
            if is_blank:
                blank_row_indices.append(row_idx)
        
        if not blank_row_indices:
            print(f"No blank rows found in sheet '{sheet_name}'.")
            return 0
        
        # Sort indices in descending order for batch deletion
        blank_row_indices.sort(reverse=True)
        
        # Create batch delete requests (delete rows from bottom to top)
        requests = []
        for row_idx in blank_row_indices:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": row_idx,
                        "endIndex": row_idx + 1
                    }
                }
            })
        
        # Execute batch deletion
        delete_request = {"requests": requests}
        
        sheet_service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body=delete_request
        ).execute()
        
        print(f"Deleted {len(blank_row_indices)} blank rows from sheet '{sheet_name}'.")
        return len(blank_row_indices)
            
    except Exception as e:
        print(f"Error processing sheet '{sheet_name}': {e}")
        return 0
    
delete_all_blank_rows(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")

#### uID

In [ ]:
def add_uid_column(sheet_service, spreadsheet_id, sheet_name, uid_col_name="uid"):
    import uuid
    import time
    from googleapiclient.errors import HttpError

    MAX_RETRIES = 3
    RETRY_DELAY = 2  # seconds
    CHUNK_SIZE = 1000

    def execute_with_retry(api_call, *args, **kwargs):
        """Execute API call with retry logic for rate limiting and transient errors"""
        for attempt in range(MAX_RETRIES):
            try:
                return api_call(*args, **kwargs).execute()
            except HttpError as e:
                if e.resp.status in [429, 500, 503]:  # Rate limit or server errors
                    if attempt < MAX_RETRIES - 1:
                        sleep_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                        print(f"Rate limited, retrying in {sleep_time}s (attempt {attempt + 1}/{MAX_RETRIES})")
                        time.sleep(sleep_time)
                        continue
                raise
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    print(f"Error occurred, retrying in {RETRY_DELAY}s (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                    time.sleep(RETRY_DELAY)
                    continue
                raise
        return None

    def column_index_to_a1(column_index):
        """Convert zero-based column index to A1 notation"""
        result = ""
        while column_index >= 0:
            result = chr(65 + (column_index % 26)) + result
            column_index = (column_index // 26) - 1
        return result

    try:
        # Get sheet metadata
        spreadsheet = execute_with_retry(
            sheet_service.spreadsheets().get,
            spreadsheetId=spreadsheet_id,
            fields="sheets(properties(sheetId,title,gridProperties(rowCount,columnCount)))"
        )

        sheet_id = None
        row_count = 0
        original_col_count = 0
        
        for sheet in spreadsheet["sheets"]:
            if sheet["properties"]["title"] == sheet_name:
                sheet_id = sheet["properties"]["sheetId"]
                row_count = sheet["properties"]["gridProperties"]["rowCount"]
                original_col_count = sheet["properties"]["gridProperties"]["columnCount"]
                break

        if sheet_id is None:
            raise ValueError(f"Sheet '{sheet_name}' not found in spreadsheet")

        # Check if UID column already exists
        result = execute_with_retry(
            sheet_service.spreadsheets().values().get,
            spreadsheetId=spreadsheet_id,
            range=f"{sheet_name}!1:1"
        )
        
        values = result.get('values', [])
        headers = values[0] if values else []
        
        # Find existing UID column
        existing_uid_col_index = None
        for i, header in enumerate(headers):
            if header == uid_col_name:
                existing_uid_col_index = i
                break

        # If UID column exists, use it instead of creating a new one
        if existing_uid_col_index is not None:
            print(f"UID column '{uid_col_name}' already exists at column {existing_uid_col_index + 1}")
            insert_col_index = existing_uid_col_index
            col_letter = column_index_to_a1(insert_col_index)
            
            # Clear existing values (keep header)
            clear_range = f"{sheet_name}!{col_letter}2:{col_letter}{row_count}"
            execute_with_retry(
                sheet_service.spreadsheets().values().clear,
                spreadsheetId=spreadsheet_id,
                range=clear_range,
                body={}
            )
        else:
            # Find the last column with data
            last_col_with_data = len(headers) if headers else 0
            
            # If we're at the grid boundary, we need to append rather than insert
            if last_col_with_data >= original_col_count:
                # Append a new column by updating the sheet's grid properties
                append_request = {
                    "updateSheetProperties": {
                        "properties": {
                            "sheetId": sheet_id,
                            "gridProperties": {
                                "columnCount": original_col_count + 1
                            }
                        },
                        "fields": "gridProperties.columnCount"
                    }
                }
                
                execute_with_retry(
                    sheet_service.spreadsheets().batchUpdate,
                    spreadsheetId=spreadsheet_id,
                    body={"requests": [append_request]}
                )
                
                insert_col_index = original_col_count
                col_letter = column_index_to_a1(insert_col_index)
                
                # Set the header for the new column
                header_range = f"{sheet_name}!{col_letter}1"
                execute_with_retry(
                    sheet_service.spreadsheets().values().update,
                    spreadsheetId=spreadsheet_id,
                    range=header_range,
                    valueInputOption="RAW",
                    body={"values": [[uid_col_name]]}
                )
            else:
                # We have room to insert a column
                insert_col_index = last_col_with_data
                col_letter = column_index_to_a1(insert_col_index)

                # Batch requests for inserting column and setting header
                batch_requests = [
                    {
                        "insertDimension": {
                            "range": {
                                "sheetId": sheet_id,
                                "dimension": "COLUMNS",
                                "startIndex": insert_col_index,
                                "endIndex": insert_col_index + 1
                            },
                            "inheritFromBefore": False
                        }
                    },
                    {
                        "updateCells": {
                            "rows": [
                                {"values": [{"userEnteredValue": {"stringValue": uid_col_name}}]}
                            ],
                            "fields": "userEnteredValue",
                            "start": {"sheetId": sheet_id, "rowIndex": 0, "columnIndex": insert_col_index}
                        }
                    }
                ]

                # Execute batch update
                execute_with_retry(
                    sheet_service.spreadsheets().batchUpdate,
                    spreadsheetId=spreadsheet_id,
                    body={"requests": batch_requests}
                )

        # Generate UUIDs and update values in chunks
        if row_count > 1:  # Only if there are data rows
            for start_row in range(1, row_count, CHUNK_SIZE):
                end_row = min(start_row + CHUNK_SIZE, row_count)
                num_rows = end_row - start_row
                
                values = [[str(uuid.uuid4())] for _ in range(num_rows)]
                
                range_uid = f"{sheet_name}!{col_letter}{start_row + 1}:{col_letter}{end_row}"
                
                body = {
                    "valueInputOption": "RAW",
                    "data": [{"range": range_uid, "values": values}]
                }

                execute_with_retry(
                    sheet_service.spreadsheets().values().batchUpdate,
                    spreadsheetId=spreadsheet_id,
                    body=body
                )
                
                # Add a small delay between chunks to avoid rate limiting
                if end_row < row_count:
                    time.sleep(1)

        print(f"UID column '{uid_col_name}' {'updated' if existing_uid_col_index is not None else 'added'} "
              f"in sheet '{sheet_name}' at column {col_letter} ({insert_col_index + 1})")
        
        return insert_col_index, col_letter

    except HttpError as e:
        print(f"Google Sheets API error: {e}")
        raise
    except Exception as e:
        print(f"Unexpected error: {e}")
        raise


# Usage example with comprehensive error handling
try:
    column_index, column_letter = add_uid_column(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")
    print(f"Operation completed successfully. UID column at {column_letter} (index {column_index})")
    
except ValueError as e:
    print(f"Configuration error: {e}")
except HttpError as e:
    if e.resp.status == 404:
        print(f"Spreadsheet or sheet not found: {e}")
    elif e.resp.status == 403:
        print(f"Permission denied: {e}")
    else:
        print(f"Google Sheets API error: {e}")
except TimeoutError:
    print("Timeout occurred. Please check your network connection and try again.")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
def get_sheet_data(sheet_service, spreadsheet_id, sheet_name):
    range_name = f"{sheet_name}"
    result = sheet_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=range_name
    ).execute()
    values = result.get('values', [])
    if not values:
        return pd.DataFrame()

    headers = values[0]
    num_cols = len(headers)

    # Truncate each row to header length, pad short rows with ''
    normalized_rows = [
        (row[:num_cols] + [''] * (num_cols - len(row)))
        for row in values[1:]
    ]

    return pd.DataFrame(normalized_rows, columns=headers)

df_target = get_sheet_data(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")
df_target = df_target.replace('', pd.NA)  # Convert empty strings to NA
df_target = df_target.replace(r'^\s*$', pd.NA, regex=True)  # Convert whitespace to NA
print("✓ Converted empty strings/whitespace to NA values")

#### Remove filters

In [ ]:
from googleapiclient.errors import HttpError

def remove_sheet_filters(spreadsheet_id, sheet_name):
    """
    Remove all filters from a specific sheet in a Google Spreadsheet
    """
    try:
        # Get spreadsheet metadata to find the sheet ID
        spreadsheet = sheet_service.spreadsheets().get(
            spreadsheetId=spreadsheet_id,
            fields="sheets(properties(sheetId,title))"
        ).execute()
        
        # Find the specific sheet ID
        sheet_id = None
        for sheet in spreadsheet['sheets']:
            if sheet['properties']['title'] == sheet_name:
                sheet_id = sheet['properties']['sheetId']
                break
        
        if not sheet_id:
            print(f"Sheet '{sheet_name}' not found")
            return False
        
        # Create request to clear filters
        requests = [{
            'clearBasicFilter': {
                'sheetId': sheet_id
            }
        }]
        
        # Execute the request
        body = {'requests': requests}
        response = sheet_service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body=body
        ).execute()
        
        print(f"✅ Successfully removed all filters from '{sheet_name}'")
        return True
        
    except HttpError as error:
        print(f"❌ An HTTP error occurred: {error}")
        return False
    except Exception as error:
        print(f"❌ An error occurred: {error}")
        return False

# Usage
SPREADSHEET_ID = ORIGINAL_SPREADSHEET_ID  
SHEET_NAME = "aligned"  

# Remove filters only
success = remove_sheet_filters(SPREADSHEET_ID, SHEET_NAME)
if success:
    print("Filters removed successfully!")
else:
    print("Failed to remove filters.")

### Refresh

In [ ]:
# Update after applying schema
def get_sheet_data(sheet_service, spreadsheet_id, sheet_name):
    range_name = f"{sheet_name}"  # Add and adjust range if needed 
    result = sheet_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=range_name
    ).execute()
    values = result.get('values', [])
    if not values:
        return pd.DataFrame()  # Return empty if no data
    return pd.DataFrame(values[1:], columns=values[0])  # Row 0 = headers

# Fetch data 
df_target = get_sheet_data(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")
df_target = df_target.replace('', pd.NA)  # Convert empty strings to NA
df_target = df_target.replace(r'^\s*$', pd.NA, regex=True)  # Convert whitespace to NA
print("✓ Converted empty strings/whitespace to NA values")

In [ ]:
def get_dataframe_summary(df_target):
    
    # --- General DataFrame Info ---
    general_info = {
        "Shape": f"{df_target.shape[0]} rows × {df_target.shape[1]} cols",
        "Memory Usage": f"{df_target.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB",
        "Columns with NA": f"{df_target.isna().any().sum()} of {len(df_target.columns)}",
        "Duplicate Rows": f"{df_target.duplicated().sum()} ({(df_target.duplicated().mean() * 100):.1f}%)",
        "Numeric Columns": f"{df_target.select_dtypes(include='number').shape[1]}",
        "Categorical Columns": f"{df_target.select_dtypes(include=['object', 'category']).shape[1]}",
        "Datetime Columns": f"{df_target.select_dtypes(include='datetime').shape[1]}"
    }

    # --- Compute dropdown options per column ---
    dropdown_options_dict = {
        col: dropdown_options.get(col, [])
        for col in df_target.columns
    }
    
    # --- Column-Level Stats ---
    column_stats = pd.DataFrame({
        'Variable': df_target.columns,
        'Dtype': df_target.dtypes.values,
        'Unique_Count': df_target.nunique().values,
        'NA_Count': df_target.isna().sum().values,
        'NA_Percentage': (df_target.isna().mean() * 100).round(1).values,
        'Duplicate_Count': df_target.apply(lambda col: col.duplicated(keep=False).sum()).values,
        'Duplicate_Percentage': (df_target.apply(lambda col: col.duplicated(keep=False).mean()) * 100).round(1).values,
        'Unique_Values': df_target.apply(lambda x: x.drop_duplicates().tolist()).values,
        'Dropdown_Options': [dropdown_options_dict[col] for col in df_target.columns],
        'Dropdown_Option_Count': [len(dropdown_options_dict[col]) for col in df_target.columns]
    }).sort_values('Unique_Count', ascending=False)
        # --- Format percentages ---
    column_stats['NA_Percentage'] = column_stats['NA_Percentage'].astype(str) + '%'
    column_stats['Duplicate_Percentage'] = column_stats['Duplicate_Percentage'].astype(str) + '%'

    return general_info, column_stats

# Run the summary
general_info, column_stats = get_dataframe_summary(df_target)

# Print General Info
print("=== GENERAL DATAFRAME INFO ===")
for key, value in general_info.items():
    print(f"{key}: {value}")


# Display Column Stats (including dropdown options)
print("\n=== COLUMN-LEVEL STATISTICS ===")
display(column_stats)

### Possible formatting

In [ ]:
# unique values in column Decision
unique_decision_values = df_target['Decision'].dropna().unique()
print(f"Unique values in 'Decision' column: {unique_decision_values}")

# unique values in column Status
unique_status_values = df_target['Status'].dropna().unique()
print(f"Unique values in 'Status' column: {unique_status_values}")

# unique values in column Deadline
unique_deadline_values = df_target['Deadline'].dropna().unique()
print(f"Unique values in 'Deadline' column: {unique_deadline_values}")

# unique values in column Application Date
unique_application_date_values = df_target['Application Date'].dropna().unique()
print(f"Unique values in 'Application Date' column: {unique_application_date_values}")

##### Decision

In [ ]:
def standardize_decision_column(sheet_service, spreadsheet_id, sheet_name="aligned"):
    """
    Standardize Decision column values:
    - 'si', 'yes', 'SI' → 'Apply'
    - 'no' → 'Skip'
    - All other values (including empty cells) → 'Unknown'
    """
    try:
        # Read all data from the sheet
        result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_name
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print(f"No data found in sheet '{sheet_name}'.")
            return False
            
        headers = values[0]
        print(f"📋 Headers found: {headers}")
        
        if 'Decision' not in headers:
            print("❌ Column 'Decision' not found in headers.")
            return False
            
        decision_idx = headers.index('Decision')
        print(f"📍 Decision column at index: {decision_idx}")
        
        # Show original values before changes
        print("\n🔍 Original Decision values (first 10 rows):")
        for i in range(1, min(11, len(values))):
            row = values[i]
            current = row[decision_idx] if len(row) > decision_idx and row[decision_idx] else '(empty)'
            print(f"  Row {i+1}: '{current}'")
        
        # Track statistics
        stats = {
            'Apply': 0,
            'Skip': 0,
            'Unknown': 0,
            'Changed': 0
        }
        
        # Process each row (skip header row)
        changes_made = False
        for i in range(1, len(values)):
            row = values[i]
            
            # Ensure row has enough columns
            while len(row) <= decision_idx:
                row.append('')
            
            # Get current decision value and clean it
            current_value = row[decision_idx] if row[decision_idx] else ''
            clean_value = str(current_value).strip().lower()
            
            # Determine new value based on rules
            if clean_value in ['si', 'yes']:
                new_value = 'Apply'
                stats['Apply'] += 1
            elif clean_value == 'no':
                new_value = 'Skip'
                stats['Skip'] += 1
            else:
                # Everything else becomes Unknown (including empty cells, random text, etc.)
                new_value = 'Unknown'
                stats['Unknown'] += 1
            
            # Update if value is different
            if row[decision_idx] != new_value:
                print(f"  Changing row {i+1}: '{row[decision_idx]}' → '{new_value}'")
                row[decision_idx] = new_value
                stats['Changed'] += 1
                changes_made = True
        
        if not changes_made:
            print("\n✅ No changes needed - Decision column already standardized")
            print(f"   Current distribution: Apply={stats['Apply']}, Skip={stats['Skip']}, Unknown={stats['Unknown']}")
            return True
        
        # Write back the entire sheet
        print(f"\n📝 Writing {len(values)} rows back to sheet '{sheet_name}'...")
        
        update_result = sheet_service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=sheet_name,
            valueInputOption="USER_ENTERED",
            body={"values": values}
        ).execute()
        
        print(f"✅ Update successful: {update_result.get('updatedCells')} cells updated")
        print(f"\n📊 Standardization complete:")
        print(f"   - 'Apply': {stats['Apply']} rows (from 'si', 'yes')")
        print(f"   - 'Skip': {stats['Skip']} rows (from 'no')")
        print(f"   - 'Unknown': {stats['Unknown']} rows (everything else including empty cells)")
        print(f"   - Total changes made: {stats['Changed']} rows")
        
        # Verify the changes by reading back
        print("\n🔍 Verifying changes (first 10 rows after update):")
        verify_result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=f"{sheet_name}!A:Z"
        ).execute()
        
        verify_values = verify_result.get('values', [])
        if len(verify_values) > 1:
            for i in range(1, min(11, len(verify_values))):
                row = verify_values[i]
                new_value = row[decision_idx] if len(row) > decision_idx and row[decision_idx] else '(empty)'
                print(f"  Row {i+1}: '{new_value}'")
        
        return True
        
    except Exception as e:
        print(f"❌ Error standardizing Decision column: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run with clear print statements to see what's happening
print("="*60)
print("🚀 STARTING DECISION COLUMN STANDARDIZATION")
print("="*60)
print("📌 Rules:")
print("   - 'si/yes' → 'Apply'")
print("   - 'no' → 'Skip'")
print("   - Everything else (including empty cells) → 'Unknown'")

standardize_decision_column(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")

print("\n" + "="*60)
print("✅ PROCESS COMPLETE - Check your Google Sheet")
print("="*60)

## Delivery

In [ ]:
# Function to duplicate an existing sheet and keep it hidden
def duplicate_sheet_hidden(sheet_service, spreadsheet_id, source_sheet_name, new_sheet_name):
    # First get the sheet ID of the source sheet
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id
    ).execute()
    
    # Check if the target sheet already exists
    target_sheet_exists = False
    source_sheet_id = None
    
    for sheet in spreadsheet['sheets']:
        if sheet['properties']['title'] == new_sheet_name:
            target_sheet_exists = True
        if sheet['properties']['title'] == source_sheet_name:
            source_sheet_id = sheet['properties']['sheetId']
    
    if source_sheet_id is None:
        raise ValueError(f"Sheet '{source_sheet_name}' not found")
    
    # If aligned sheet already exists, delete it first
    if target_sheet_exists:
        # Find the sheet ID of the existing target sheet
        target_sheet_id = None
        for sheet in spreadsheet['sheets']:
            if sheet['properties']['title'] == new_sheet_name:
                target_sheet_id = sheet['properties']['sheetId']
                break
        
        if target_sheet_id is not None:
            # Delete the existing sheet
            delete_request = {
                "requests": [
                    {
                        "deleteSheet": {
                            "sheetId": target_sheet_id
                        }
                    }
                ]
            }
            
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body=delete_request
            ).execute()
            print(f"Deleted existing sheet '{new_sheet_name}'")
    
    # Create a duplicate of the sheet
    batch_update_request = {
        "requests": [
            {
                "duplicateSheet": {
                    "sourceSheetId": source_sheet_id,
                    "insertSheetIndex": 8,  # Position where to insert the new sheet
                    "newSheetName": new_sheet_name
                }
            }
        ]
    }
    
    # Execute the batch update to duplicate the sheet
    response = sheet_service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body=batch_update_request
    ).execute()
    
    # Get the ID of the newly created sheet
    new_sheet_id = response['replies'][0]['duplicateSheet']['properties']['sheetId']
    
    # Update the sheet properties to hide it
    hide_request = {
        "requests": [
            {
                "updateSheetProperties": {
                    "properties": {
                        "sheetId": new_sheet_id,
                        "hidden": False  # Set to True to hide the sheet
                    },
                    "fields": "hidden"
                }
            }
        ]
    }
    
    # Execute the batch update to hide the sheet
    sheet_service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body=hide_request
    ).execute()
    
    print(f"Duplicated sheet '{source_sheet_name}' to hidden sheet '{new_sheet_name}'")
    return response

# Usage
duplicate_sheet_hidden(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned", "delivery")

# End

In [ ]:
end_time = time.time()
end_readable = time.ctime(end_time)
execution_time = (end_time - start_time)/60


print(f"Start: {formatted_start_time}")
print(f"End: {end_readable}")
print(f"Execution time: {execution_time:.2f} minutes")

print("="*80)